# 00b — Compute recurrence frontloading \(F\) from the frozen 384-token prefix

This is the cleaned, optimized reproduction of the historical `minicache.ipynb` recurrence extractor.

The crucial historical preprocessing step is:

1. tokenize the **raw document** with `Qwen/Qwen3-0.6B-Base`;
2. require at least 384 Qwen tokens;
3. take exactly the first 384 token IDs;
4. decode those IDs back to text with `skip_special_tokens=True`;
5. compute sentence segmentation, lexical recurrence, shuffle correction, and \(F\) on that decoded prefix text.

The earlier reproduction notebook incorrectly computed recurrence over the full raw document. This version matches the historical pipeline.

Frozen settings:
- scorer/tokenizer: `Qwen/Qwen3-0.6B-Base`
- prefix length: 384 Qwen tokens
- sentence segmentation: NLTK `sent_tokenize`
- content words: scikit-learn `ENGLISH_STOP_WORDS`
- word regex: `[a-zA-Z][a-zA-Z'-]*`
- minimum content-word length: 3
- bins: `(0,32)`, `(33,64)`, `(65,96)`, `(97,128)`, `(129,160)`,
  `(161,192)`, `(193,224)`, `(225,256)`, `(257,320)`
- 200 sentence-order shuffles
- shuffle seed: 42
- near = first three bins
- far = last three bins
- \(F=R_{\rm near}-R_{\rm far}\)

The shuffle computation is optimized by precomputing the pairwise Jaccard matrix once per document. This is mathematically identical to the historical slow implementation.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from transformers import AutoTokenizer

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd

DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA / "raw" / "ghostbuster_detector_pool.parquet"
ID_COL = "row_id"
TEXT_COL = "text"

OUTPUT_PATH = RESULTS / "ghostbuster_detector_frontloading.parquet"

TOKENIZER_NAME = "Qwen/Qwen3-0.6B-Base"
PREFIX_LEN = 384

TOKEN_DISTANCE_BINS = [
    (0, 32),
    (33, 64),
    (65, 96),
    (97, 128),
    (129, 160),
    (161, 192),
    (193, 224),
    (225, 256),
    (257, 320),
]

NEAR_TOKEN_BINS = TOKEN_DISTANCE_BINS[:3]
FAR_TOKEN_BINS = TOKEN_DISTANCE_BINS[-3:]

N_SHUFFLES = 200
SHUFFLE_SEED = 42

raw = pd.read_parquet(INPUT_PATH)

assert ID_COL in raw.columns
assert TEXT_COL in raw.columns
assert not raw[ID_COL].duplicated().any()

print("ROOT:", ROOT)
print("input:", INPUT_PATH)
print("output:", OUTPUT_PATH)
print("rows:", len(raw))

/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: ..
input: ../data/raw/ghostbuster_detector_pool.parquet
output: ../results/ghostbuster_detector_frontloading.parquet
rows: 9000


In [2]:
nltk.download("punkt", quiet=True)

try:
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

WORD_RE = re.compile(
    r"[a-zA-Z][a-zA-Z'-]*"
)
STOPWORDS = set(ENGLISH_STOP_WORDS)

/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/user/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


## Frozen 384-token prefix

This matches the historical `qwen_prefix_text`/`prefix_text` construction. Recurrence is measured on the **decoded 384-token prefix**, not on the full raw document.

In [3]:
def qwen_prefix_text(
    text,
    prefix_len=PREFIX_LEN,
):
    ids = tokenizer(
        str(text),
        add_special_tokens=False,
    )["input_ids"]

    if len(ids) < prefix_len:
        return None

    ids = ids[:prefix_len]

    return tokenizer.decode(
        ids,
        skip_special_tokens=True,
    )

## Sentence lexical sets and approximate token centers

After the prefix has been decoded, sentence centers are reconstructed by tokenizing each segmented sentence independently and accumulating its Qwen-token length. This matches the historical implementation.

In [4]:
def content_words(sentence):
    return {
        w
        for w in WORD_RE.findall(
            sentence.lower()
        )
        if (
            w not in STOPWORDS
            and len(w) >= 3
        )
    }


def sentence_sets_and_centers(text):
    sents = [
        s.strip()
        for s in sent_tokenize(text)
        if s.strip()
    ]

    word_sets = []
    centers = []

    cursor = 0

    for sent in sents:
        ids = tokenizer(
            sent,
            add_special_tokens=False,
        )["input_ids"]

        start = cursor
        end = cursor + len(ids)

        centers.append(
            (start + end) / 2
        )

        word_sets.append(
            content_words(sent)
        )

        cursor = end

    return (
        word_sets,
        np.asarray(
            centers,
            dtype=float,
        ),
    )

## Optimized pairwise recurrence computation

The historical implementation recomputed Jaccard overlap after every shuffle. Because shuffling only changes which lexical sentence occupies each fixed positional slot, the pairwise Jaccard matrix can be computed once and reindexed for each permutation without changing the statistic.

In [5]:
def jaccard_matrix(word_sets):
    n = len(word_sets)

    J = np.full(
        (n, n),
        np.nan,
        dtype=np.float64,
    )

    for i in range(n):
        J[i, i] = 1.0
        a = word_sets[i]

        for j in range(i + 1, n):
            b = word_sets[j]
            union = a | b

            value = (
                np.nan
                if not union
                else len(a & b) / len(union)
            )

            J[i, j] = value
            J[j, i] = value

    return J


def bin_pair_indices(centers):
    n = len(centers)
    result = {}

    for lo, hi in TOKEN_DISTANCE_BINS:
        ii = []
        jj = []

        for i in range(n):
            for j in range(i + 1, n):
                distance = (
                    centers[j]
                    - centers[i]
                )

                if lo <= distance <= hi:
                    ii.append(i)
                    jj.append(j)

        result[(lo, hi)] = (
            np.asarray(ii, dtype=np.int32),
            np.asarray(jj, dtype=np.int32),
        )

    return result

In [6]:
def observed_from_matrix(
    J,
    pair_idx,
):
    result = {}

    for b in TOKEN_DISTANCE_BINS:
        ii, jj = pair_idx[b]

        if len(ii) == 0:
            result[b] = np.nan
            continue

        values = J[ii, jj]
        values = values[
            np.isfinite(values)
        ]

        result[b] = (
            float(values.mean())
            if len(values)
            else np.nan
        )

    return result


def shuffled_from_matrix(
    J,
    pair_idx,
    n_shuffles=N_SHUFFLES,
    seed=SHUFFLE_SEED,
):
    n = J.shape[0]

    rng = np.random.default_rng(
        seed
    )

    totals = {
        b: 0.0
        for b in TOKEN_DISTANCE_BINS
    }

    counts = {
        b: 0
        for b in TOKEN_DISTANCE_BINS
    }

    for _ in range(n_shuffles):
        order = rng.permutation(n)

        for b in TOKEN_DISTANCE_BINS:
            ii, jj = pair_idx[b]

            if len(ii) == 0:
                continue

            values = J[
                order[ii],
                order[jj],
            ]

            values = values[
                np.isfinite(values)
            ]

            if len(values):
                totals[b] += float(
                    values.mean()
                )
                counts[b] += 1

    return {
        b: (
            totals[b] / counts[b]
            if counts[b] > 0
            else np.nan
        )
        for b in TOKEN_DISTANCE_BINS
    }

In [7]:
def safe_mean(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    valid = np.isfinite(values)

    if not valid.any():
        return np.nan

    return float(
        values[valid].mean()
    )


def compute_minicache_features(
    raw_text,
):
    prefix_text = qwen_prefix_text(
        raw_text,
        prefix_len=PREFIX_LEN,
    )

    result = {
        "eligible_384": prefix_text is not None,
    }

    if prefix_text is None:
        result["n_sentences"] = np.nan

        for lo, hi in TOKEN_DISTANCE_BINS:
            result[
                f"token_excess_{lo}_{hi}"
            ] = np.nan

        result["near_token_cache"] = np.nan
        result["far_token_cache"] = np.nan
        result["token_frontloading"] = np.nan

        return result

    word_sets, centers = (
        sentence_sets_and_centers(
            prefix_text
        )
    )

    result["n_sentences"] = len(
        word_sets
    )

    if len(word_sets) < 2:
        for lo, hi in TOKEN_DISTANCE_BINS:
            result[
                f"token_excess_{lo}_{hi}"
            ] = np.nan

        result["near_token_cache"] = np.nan
        result["far_token_cache"] = np.nan
        result["token_frontloading"] = np.nan

        return result

    J = jaccard_matrix(word_sets)
    pair_idx = bin_pair_indices(
        centers
    )

    observed = observed_from_matrix(
        J,
        pair_idx,
    )

    shuffled = shuffled_from_matrix(
        J,
        pair_idx,
        n_shuffles=N_SHUFFLES,
        seed=SHUFFLE_SEED,
    )

    for lo, hi in TOKEN_DISTANCE_BINS:
        o = observed[(lo, hi)]
        s = shuffled[(lo, hi)]

        result[
            f"token_excess_{lo}_{hi}"
        ] = (
            o - s
            if (
                np.isfinite(o)
                and np.isfinite(s)
            )
            else np.nan
        )

    near_values = [
        result[
            f"token_excess_{lo}_{hi}"
        ]
        for lo, hi in NEAR_TOKEN_BINS
    ]

    far_values = [
        result[
            f"token_excess_{lo}_{hi}"
        ]
        for lo, hi in FAR_TOKEN_BINS
    ]

    result[
        "near_token_cache"
    ] = safe_mean(
        near_values
    )

    result[
        "far_token_cache"
    ] = safe_mean(
        far_values
    )

    if (
        np.isfinite(
            result["near_token_cache"]
        )
        and np.isfinite(
            result["far_token_cache"]
        )
    ):
        result[
            "token_frontloading"
        ] = (
            result["near_token_cache"]
            - result["far_token_cache"]
        )
    else:
        result[
            "token_frontloading"
        ] = np.nan

    return result

## Single-document sanity check

In [8]:
example = raw.iloc[0]

example_features = (
    compute_minicache_features(
        example[TEXT_COL]
    )
)

print("ID:", example[ID_COL])

for key, value in example_features.items():
    print(
        f"{key:28s}",
        value,
    )

ID: original::wp::gpt::1
eligible_384                 True
n_sentences                  18
token_excess_0_32            -0.0019895487722822614
token_excess_33_64           -0.009482692581981787
token_excess_65_96           0.007679824650077837
token_excess_97_128          0.0015186091906945112
token_excess_129_160         0.005946166317685976
token_excess_161_192         -0.00035322244232967835
token_excess_193_224         0.0039034143485133736
token_excess_225_256         -0.012841765544685768
token_excess_257_320         0.0045642476804708485
near_token_cache             -0.0012641389013954036
far_token_cache              -0.0014580345052338489
token_frontloading           0.0001938956038384453


## Optional historical edge-case check

If the Ghostbuster detector pool contains the Reuters document that exposed the preprocessing mismatch, this cell prints its fresh value. Under the correct 384-token prefix pipeline it should agree with the historical recurrence artifact.

In [9]:
EDGE_FILE = "reuter/human/JimGilchrist/9.txt"

if "file" in raw.columns and (
    raw["file"] == EDGE_FILE
).any():
    edge = raw.loc[
        raw["file"] == EDGE_FILE
    ].iloc[0]

    edge_features = (
        compute_minicache_features(
            edge[TEXT_COL]
        )
    )

    print("edge file:", EDGE_FILE)
    print(
        "token_frontloading:",
        edge_features[
            "token_frontloading"
        ],
    )

    for lo, hi in TOKEN_DISTANCE_BINS:
        print(
            f"{lo:3d}-{hi:3d}:",
            edge_features[
                f"token_excess_{lo}_{hi}"
            ],
        )

edge file: reuter/human/JimGilchrist/9.txt
token_frontloading: -0.0467028985507246
  0- 32: nan
 33- 64: 0.023832815734989687
 65- 96: nan
 97-128: nan
129-160: nan
161-192: -0.094368530020704
193-224: 0.07053571428571428
225-256: nan
257-320: nan


## Full-corpus extraction

In [10]:
rows = []

for row_id, text in tqdm(
    raw[
        [ID_COL, TEXT_COL]
    ].itertuples(
        index=False,
        name=None,
    ),
    total=len(raw),
    desc="Recurrence/frontloading",
):
    rows.append({
        ID_COL: row_id,
        **compute_minicache_features(
            text
        ),
    })

features = pd.DataFrame(
    rows
)

features.to_parquet(
    OUTPUT_PATH,
    index=False,
)

print("saved:", OUTPUT_PATH)
print("shape:", features.shape)

display(
    features.head()
)

display(
    features[
        [
            "near_token_cache",
            "far_token_cache",
            "token_frontloading",
        ]
    ].describe()
)

Recurrence/frontloading: 100%|██████████████████████████████████████████████████████████████████████| 9000/9000 [01:32<00:00, 97.16it/s]

saved: ../results/ghostbuster_detector_frontloading.parquet
shape: (9000, 15)


,row_id,eligible_384,n_sentences,token_excess_0_32,token_excess_33_64,token_excess_65_96,token_excess_97_128,token_excess_129_160,token_excess_161_192,token_excess_193_224,token_excess_225_256,token_excess_257_320,near_token_cache,far_token_cache,token_frontloading
0,original::wp::gpt::1,True,18.0,-0.001990,-0.009483,0.007680,0.001519,0.005946,-0.000353,0.003903,-0.012842,0.004564,-0.001264,-0.001458,0.000194
1,original::wp::gpt::2,True,18.0,-0.004537,0.001152,0.005287,-0.003132,0.002600,-0.002244,0.004633,-0.000008,-0.007369,0.000634,-0.000915,0.001549
2,original::wp::gpt::3,True,25.0,0.000365,-0.002147,-0.004197,0.013826,-0.003695,-0.002460,-0.001986,-0.004942,-0.001519,-0.001993,-0.002816,0.000823
3,original::wp::gpt::4,True,25.0,0.007384,-0.000159,0.001805,-0.006018,-0.001257,0.006141,-0.006685,-0.002053,-0.003645,0.003010,-0.004128,0.007137
4,original::wp::gpt::5,True,26.0,0.002967,-0.000026,0.000576,0.000383,-0.002055,-0.006446,0.004909,0.001156,-0.003024,0.001173,0.001014,0.000159


,near_token_cache,far_token_cache,token_frontloading
count,7798.000000,7798.000000,7798.000000
mean,0.000616,-0.000094,0.000710
std,0.007348,0.010188,0.015554
min,-0.037297,-0.065953,-0.082471
25%,-0.003468,-0.005593,-0.007614
50%,0.000136,-0.000545,0.000448
75%,0.004090,0.005083,0.008499
max,0.057647,0.070536,0.123600


In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

# Resolve reproduction root.
HERE = Path.cwd().resolve()

candidates = [
    HERE,
    HERE.parent,
    HERE.parent.parent,
]

REPRO_ROOT = None

for p in candidates:
    if (p / "results" / "frontloading_same_family_pairs.csv").exists():
        REPRO_ROOT = p
        break

    if (p / "reproduction" / "results" / "frontloading_same_family_pairs.csv").exists():
        REPRO_ROOT = p / "reproduction"
        break

if REPRO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate reproduction/results/frontloading_same_family_pairs.csv"
    )

print("REPRO_ROOT:", REPRO_ROOT)

PAIRS_PATH = (
    REPRO_ROOT
    / "results"
    / "frontloading_same_family_pairs.csv"
)

RAID_DIR = (
    REPRO_ROOT
    / "data"
    / "external"
    / "raid_parquet"
    / "raid"
    / "partial-train"
)

pair_df = pd.read_csv(PAIRS_PATH)

print("saved pair rows:", len(pair_df))
display(pair_df.head())

# We need both base and chat documents for every source/family pair.
family_models = {
    "mistral": ("mistral", "mistral-chat"),
    "mpt": ("mpt", "mpt-chat"),
    "cohere": ("cohere", "cohere-chat"),
}

parquet_files = sorted(RAID_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No RAID parquet files found under {RAID_DIR}"
    )

paths_sql = "[" + ",".join(
    "'" + str(p).replace("'", "''") + "'"
    for p in parquet_files
) + "]"

wanted_source_ids = pair_df["source_id"].astype(str).unique().tolist()

# DuckDB list literal for exact source IDs.
source_sql = "[" + ",".join(
    "'" + s.replace("'", "''") + "'"
    for s in wanted_source_ids
) + "]"

wanted_models = [
    "mistral",
    "mistral-chat",
    "mpt",
    "mpt-chat",
    "cohere",
    "cohere-chat",
]

models_sql = "(" + ",".join(
    "'" + m + "'"
    for m in wanted_models
) + ")"

con = duckdb.connect()

query = f"""
SELECT
    source_id,
    model,
    domain,
    decoding,
    repetition_penalty,
    attack,
    generation
FROM read_parquet({paths_sql}, union_by_name=true)
WHERE CAST(source_id AS VARCHAR) IN (
    SELECT * FROM UNNEST({source_sql})
)
AND model IN {models_sql}
AND domain = 'news'
AND lower(CAST(decoding AS VARCHAR)) = 'greedy'
AND lower(CAST(repetition_penalty AS VARCHAR)) = 'no'
AND lower(CAST(attack AS VARCHAR)) = 'none'
"""

docs = con.execute(query).fetchdf()

print("loaded source/model docs:", len(docs))

# Tag family.
model_to_family = {
    "mistral": "mistral",
    "mistral-chat": "mistral",
    "mpt": "mpt",
    "mpt-chat": "mpt",
    "cohere": "cohere",
    "cohere-chat": "cohere",
}

docs["family"] = docs["model"].map(model_to_family)

display(
    docs.groupby(["family", "model"])
    .size()
    .rename("n")
    .to_frame()
)

# Attach the saved standalone 03b F values.
saved_long_parts = []

for family, (base_model, chat_model) in family_models.items():
    sub = pair_df[pair_df["family"] == family].copy()

    base = sub[
        ["family", "domain", "source_id", "F_base"]
    ].rename(columns={"F_base": "F_03b"})
    base["model"] = base_model

    chat = sub[
        ["family", "domain", "source_id", "F_chat"]
    ].rename(columns={"F_chat": "F_03b"})
    chat["model"] = chat_model

    saved_long_parts.extend([base, chat])

saved_long = pd.concat(saved_long_parts, ignore_index=True)

verify = docs.merge(
    saved_long,
    on=["family", "domain", "source_id", "model"],
    how="inner",
    validate="one_to_one",
)

print("documents to verify:", len(verify))
display(verify.head())

REPRO_ROOT: ..
saved pair rows: 719


,family,domain,source_id,F_base,F_chat,delta_chat_minus_base
0,mpt,news,01496450-2675-4e31-8ab6-da63b43d32e2,0.138360,-0.013764,-0.152125
1,mpt,news,02daa11f-9f13-4b52-8f0f-67695b40576c,0.159840,0.004813,-0.155027
2,mpt,news,07b62efc-2258-427e-8a40-5ae1cf6ecbd9,0.129593,-0.032637,-0.162230
3,mpt,news,0d4538fa-d0bc-4c20-a5d0-183248c6fbd7,-0.013109,-0.029303,-0.016194
4,mpt,news,0f620d36-00b3-44d0-b5d8-67ffdd39f97d,0.249981,-0.014042,-0.264023


loaded source/model docs: 3654


n
family  model            
cohere  cohere        609
        cohere-chat   609
mistral mistral       609
        mistral-chat  609
mpt     mpt           609
        mpt-chat      609

documents to verify: 1438


,source_id,model,domain,decoding,repetition_penalty,attack,generation,family,F_03b
0,d1e2651e-d549-455f-94f9-5cec1a8c9d0f,mistral,news,greedy,no,none,"""Britain's world 100m champion Dwain Chambers ...",mistral,0.654577
1,4eb61c3d-6be1-4370-bb95-d9103cf692ae,mistral,news,greedy,no,none,The 2017 World Athletics Championships in Lond...,mistral,-0.097967
2,85e87131-78fd-4613-88df-64787fc8be98,mistral,news,greedy,no,none,England and British and Irish Lions flanker Ja...,mistral,0.136703
3,21be9fa4-8262-450d-9c37-58d45f35f3ee,mistral,news,greedy,no,none,Manchester United manager Sir Alex Ferguson ha...,mistral,0.252493
4,a605a524-2367-4022-affc-9e68a35beadf,mistral,news,greedy,no,none,"""England fast bowler James Anderson is confide...",mistral,0.154875



## Same-family verification with the exact frozen recurrence extractor

The cells below reuse the exact `compute_minicache_features` implementation defined in this notebook. This function applies the frozen 384-Qwen-token prefix rule and returns the paper's recurrence frontloading scalar as `token_frontloading`.

The comparison checks the standalone `03b` values document-by-document and then recomputes the same-family paired effects from the exact frozen extractor.


In [12]:

# Exact frozen 00b extractor defined above.
frozen_frontloading_fn = compute_minicache_features

def extract_F(result):
    return float(result["token_frontloading"])

F_frozen = []

for i, text in enumerate(verify["generation"], start=1):
    result = frozen_frontloading_fn(text)
    F_frozen.append(extract_F(result))

    if i % 100 == 0:
        print(f"verified {i}/{len(verify)}")

verify["F_00b"] = F_frozen
verify["abs_error"] = (verify["F_00b"] - verify["F_03b"]).abs()
verify["signed_error"] = verify["F_00b"] - verify["F_03b"]

finite = verify[["F_00b", "F_03b"]].dropna()

print("\nDOCUMENT-LEVEL VERIFICATION")
print("---------------------------")
print("n total =", len(verify))
print("n finite =", len(finite))
print("MAD =", verify["abs_error"].mean())
print("median abs error =", verify["abs_error"].median())
print("max abs error =", verify["abs_error"].max())

if len(finite) >= 2:
    corr = finite.corr().iloc[0, 1]
else:
    corr = np.nan
print("corr =", corr)

display(
    verify.groupby(["family", "model"])
    .agg(
        n=("source_id", "size"),
        n_finite_00b=("F_00b", "count"),
        mean_F_03b=("F_03b", "mean"),
        mean_F_00b=("F_00b", "mean"),
        mean_abs_error=("abs_error", "mean"),
        max_abs_error=("abs_error", "max"),
    )
)

# Recompute exact frozen paired effects using only sources with finite F on both sides.
frozen_pairs = []

for family, (base_model, chat_model) in family_models.items():
    sub = verify[verify["family"] == family]

    wide = sub.pivot(
        index="source_id",
        columns="model",
        values="F_00b",
    )

    if base_model not in wide.columns or chat_model not in wide.columns:
        print(f"{family}: missing one side after exact 00b extraction")
        continue

    wide = wide[[base_model, chat_model]].dropna()

    if len(wide) == 0:
        print(f"{family}: no finite matched pairs after exact 00b extraction")
        continue

    delta = wide[chat_model] - wide[base_model]
    x = delta.to_numpy()

    rng = np.random.default_rng(42)
    idx = rng.integers(
        0,
        len(x),
        size=(5000, len(x)),
    )
    boot = x[idx].mean(axis=1)
    lo, hi = np.quantile(boot, [0.025, 0.975])

    frozen_pairs.append({
        "family": family,
        "n": len(x),
        "F_base": wide[base_model].mean(),
        "F_chat": wide[chat_model].mean(),
        "delta_chat_minus_base": x.mean(),
        "ci_low": lo,
        "ci_high": hi,
    })

frozen_summary = pd.DataFrame(frozen_pairs)

print("\nEXACT FROZEN 00b SAME-FAMILY EFFECTS")
display(frozen_summary)

doc_out = (
    REPRO_ROOT
    / "results"
    / "frontloading_same_family_00b_document_comparison.csv"
)
summary_out = (
    REPRO_ROOT
    / "results"
    / "frontloading_same_family_00b_verification.csv"
)

verify.to_csv(doc_out, index=False)
frozen_summary.to_csv(summary_out, index=False)

print("\nSaved:")
print(doc_out)
print(summary_out)


verified 100/1438
verified 200/1438
verified 300/1438
verified 400/1438
verified 500/1438
verified 600/1438
verified 700/1438
verified 800/1438
verified 900/1438
verified 1000/1438
verified 1100/1438
verified 1200/1438
verified 1300/1438
verified 1400/1438

DOCUMENT-LEVEL VERIFICATION
---------------------------
n total = 1438
n finite = 1065
MAD = 0.039647995314159086
median abs error = 0.01679463727276149
max abs error = 0.43273864507670845
corr = 0.8722005137333351


n  n_finite_00b  mean_F_03b  mean_F_00b  \
family  model                                                     
cohere  cohere        300           281    0.003337    0.000308   
        cohere-chat   300           242    0.002818    0.002409   
mistral mistral       337           320    0.180214    0.123079   
        mistral-chat  337           123   -0.007187   -0.000595   
mpt     mpt            82            82    0.180967    0.155967   
        mpt-chat       82            17   -0.021179   -0.005319   

                      mean_abs_error  max_abs_error  
family  model                                        
cohere  cohere              0.012359       0.068784  
        cohere-chat         0.012196       0.059571  
mistral mistral             0.079897       0.432739  
        mistral-chat        0.020983       0.233793  
mpt     mpt                 0.089409       0.260340  
        mpt-chat            0.018883       0.059281


EXACT FROZEN 00b SAME-FAMILY EFFECTS


,family,n,F_base,F_chat,delta_chat_minus_base,ci_low,ci_high
0,mistral,115,0.104419,-0.000325,-0.104744,-0.134444,-0.076023
1,mpt,17,0.174528,-0.005319,-0.179847,-0.253213,-0.106362
2,cohere,225,0.001000,0.002230,0.001230,-0.001884,0.004286



Saved:
../results/frontloading_same_family_00b_document_comparison.csv
../results/frontloading_same_family_00b_verification.csv
